Chronos (Theil's u, pearson, bias)

In [ ]:
!pip install git+https://github.com/amazon-science/chronos-forecasting.git

  Cloning https://github.com/amazon-science/chronos-forecasting.git to /tmp/pip-req-build-p01tr15a
  Running command git clone --filter=blob:none --quiet https://github.com/amazon-science/chronos-forecasting.git /tmp/pip-req-build-p01tr15a
  Resolved https://github.com/amazon-science/chronos-forecasting.git to commit 6a9c8dadac04eb85befc935043e3e2cce914267f
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 107.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 82.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 46.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━

chronos (theil's u, pearson, bias)

In [ ]:
import ast
import numpy as np
import pandas as pd
from chronos import ChronosPipeline
import torch
from scipy.stats import pearsonr
import warnings
warnings.filterwarnings("ignore")

# === Load and Parse the ESG Data ===
with open('/content/PaperReady_numeric_esg.txt', 'r') as f:
    content = f.read()
    start = content.find('{')
    end = content.rfind('}') + 1
    dict_str = content[start:end]
    numeric_esg = ast.literal_eval(dict_str)

# === Load Chronos Model ===
pretrained_model = ChronosPipeline.from_pretrained("amazon/chronos-t5-small", device_map="auto")

# === Define Metrics Storage ===
metrics = {
    "Theil's U": {},
    'Pearson': {},
    'Bias': {}
}
window_size = 8  # Retain window size logic

# === Loop Through Each Company ===
for ticker, series in numeric_esg.items():
    if len(series) < window_size * 2:
        for metric in metrics:
            metrics[metric][ticker] = np.nan
        continue

    series = np.array(series, dtype=float)
    n = len(series)
    split_idx = int(n * 0.75)
    train, test = series[:split_idx], series[split_idx:]

    # === Build sliding window from train data ===
    input_windows = []
    for i in range(len(train) - window_size + 1):
        input_windows.append(train[i:i + window_size])
    if not input_windows:
        for metric in metrics:
            metrics[metric][ticker] = np.nan
        continue

    # === Use last window from train ===
    input_tensor = torch.tensor(input_windows[-1], dtype=torch.float32).unsqueeze(0)
    prediction_length = len(test)

    try:
        forecast = pretrained_model.predict(input_tensor, prediction_length=prediction_length)
        y_pred = forecast[0].mean(dim=-1).detach().cpu().numpy()
        y_true = np.array(test)

        min_len = min(len(y_pred), len(y_true))
        y_pred = y_pred[:min_len]
        y_true = y_true[:min_len]

        # === Theil's U
        numerator = np.sqrt(np.sum((y_pred - y_true) ** 2))
        denominator = np.sqrt(np.sum(y_pred ** 2)) + np.sqrt(np.sum(y_true ** 2))
        theils_u = numerator / denominator if denominator != 0 else np.nan

        # === Pearson
        if np.std(y_pred) == 0 or np.std(y_true) == 0:
            pearson_corr = np.nan
        else:
            pearson_corr, _ = pearsonr(y_true, y_pred)

        # === Bias
        bias = np.mean(y_pred - y_true)

        # === Store metrics
        metrics["Theil's U"][ticker] = theils_u
        metrics["Pearson"][ticker] = pearson_corr
        metrics["Bias"][ticker] = bias

        # Print per-company
        theils_u_str = f"{theils_u:.4f}" if not np.isnan(theils_u) else "NaN"
        pearson_str = f"{pearson_corr:.4f}" if not np.isnan(pearson_corr) else "NaN"
        bias_str = f"{bias:.4f}" if not np.isnan(bias) else "NaN"

        print(f"{ticker}: Theil's U = {theils_u_str}, Pearson = {pearson_str}, Bias = {bias_str}")

    except Exception as e:
        print(f"❌ Error for {ticker}: {e}")
        for metric in metrics:
            metrics[metric][ticker] = np.nan
        continue

# === Print average of new metrics
print("\n📊 Average Metrics Across Companies:")
for metric, values in metrics.items():
    valid = [v for v in values.values() if not np.isnan(v)]
    avg = np.mean(valid) if valid else np.nan
    print(f"{metric}: {avg:.4f}")

# === Save to Excel
results_df = pd.DataFrame(metrics)
results_df.to_excel("chronos_theils_pearson_bias_window8.xlsx", index=True)
print("\n✅ Results saved to 'chronos_theils_pearson_bias_window8.xlsx'")


A: Theil's U = 0.4530, Pearson = 0.3808, Bias = 13.4234
AAPL: Theil's U = 0.8729, Pearson = -0.1104, Bias = 50.7290
ABBV: Theil's U = 0.2141, Pearson = -0.3464, Bias = 7.9784
ABT: Theil's U = 0.1526, Pearson = 0.1694, Bias = 6.9193
ACGL: Theil's U = 0.1151, Pearson = -0.0303, Bias = 3.4475
ACN: Theil's U = 0.5588, Pearson = -0.0804, Bias = 16.8611
ADBE: Theil's U = 0.4252, Pearson = 0.2828, Bias = 10.3966
ADI: Theil's U = 0.3621, Pearson = -0.1250, Bias = 12.5422
ADM: Theil's U = 0.2602, Pearson = 0.0447, Bias = -3.5132
ADP: Theil's U = 0.3372, Pearson = 0.4546, Bias = 5.5660
ADSK: Theil's U = 0.3801, Pearson = 0.1663, Bias = 10.1137
AEE: Theil's U = 0.2196, Pearson = 0.1732, Bias = 12.4752
AEP: Theil's U = 0.3879, Pearson = -0.0488, Bias = -4.1468
AES: Theil's U = 0.1862, Pearson = -0.1622, Bias = 7.3778
AFL: Theil's U = 0.3837, Pearson = -0.3549, Bias = 10.4415
AIG: Theil's U = 0.2513, Pearson = -0.3214, Bias = 9.1924
AIZ: Theil's U = 0.2158, Pearson = -0.2344, Bias = 9.0399
AJG: The

chronos (all metrics)

In [ ]:
import ast
import numpy as np
from sklearn.metrics import mean_squared_error, mean_absolute_error
import pandas as pd
from chronos import ChronosPipeline
import torch
from scipy.stats import pearsonr

def safe_float_format(value, fmt='.4f'):
    """Safely format numeric values handling NaN/None cases"""
    if value is None or np.isnan(value):
        return 'NaN'
    return f"{value:{fmt}}"

# === Load and Parse the ESG Data ===
with open('/content/PaperReady_numeric_esg.txt', 'r') as f:
    content = f.read()
    start = content.find('{')
    end = content.rfind('}') + 1
    dict_str = content[start:end]
    numeric_esg = ast.literal_eval(dict_str)

print(f"Total companies loaded: {len(numeric_esg)}")

# === Load Chronos Model ===
pretrained_model = ChronosPipeline.from_pretrained("amazon/chronos-t5-small", device_map="auto")

# === Define Metrics Storage ===
metrics = {
    'MSE': {}, 'MAE': {}, 'RMSE': {},
    'Error Rate (%)': {}, 'MAPE (%)': {}, 'WQL': {},
    'Theil\'s U': {}, 'Pearson': {}, 'Bias': {}
}
wql_var_threshold = 1e-6
window_size = 8

def calculate_theils_u(y_true, y_pred):
    """Calculate Theil's U with safe division"""
    try:
        return np.sqrt(np.sum((y_pred - y_true) ** 2)) / (
            np.sqrt(np.sum(y_pred ** 2)) + np.sqrt(np.sum(y_true ** 2))
        )
    except:
        return np.nan

def calculate_error_rate(y_true, y_pred):
    """Calculate error rate with safe division"""
    with np.errstate(divide='ignore', invalid='ignore'):
        abs_true = np.abs(y_true)
        errors = np.abs(y_true - y_pred) / np.where(abs_true == 0, np.nan, abs_true)
        return 100 * np.nanmean(errors)

# === Tracking ===
processed = 0
skipped = 0

# === Loop Through Each Company ===
for ticker, series in numeric_esg.items():
    if len(series) < window_size * 2:
        skipped += 1
        for metric in metrics:
            metrics[metric][ticker] = np.nan
        continue

    try:
        series = np.array(series, dtype=float)
        n = len(series)
        split_idx = int(n * 0.75)
        train, test = series[:split_idx], series[split_idx:]

        if len(test) == 0:
            raise ValueError("Test set is empty")

        # Use sliding window on training data
        input_windows = []
        for i in range(len(train) - window_size + 1):
            input_windows.append(train[i:i + window_size])

        if not input_windows:
            raise ValueError("No input windows created")

        # Predict using the last window
        input_tensor = torch.tensor(input_windows[-1], dtype=torch.float32).unsqueeze(0)
        forecast = pretrained_model.predict(input_tensor, prediction_length=len(test))
        y_pred = forecast[0, :, 0].detach().cpu().numpy()
        y_true = test.copy()

        # Ensure equal lengths
        min_len = min(len(y_pred), len(y_true))
        if min_len == 0:
            raise ValueError("Empty prediction or test set")

        y_pred = y_pred[:min_len]
        y_true = y_true[:min_len]

        # === Core Metrics Calculation ===
        mse = mean_squared_error(y_true, y_pred)
        mae = mean_absolute_error(y_true, y_pred)
        rmse = np.sqrt(mse)
        error_rate = calculate_error_rate(y_true, y_pred)
        mape = 100 * np.nanmean(np.abs((y_true - y_pred) / np.where(y_true == 0, np.nan, y_true)))
        variance = np.var(y_true)
        wql = np.nan if variance < wql_var_threshold else mse / variance

        # === Additional Metrics ===
        theils_u = calculate_theils_u(y_true, y_pred)

        if np.std(y_true) == 0 or np.std(y_pred) == 0:
            pearson_corr = np.nan
        else:
            pearson_corr, _ = pearsonr(y_true, y_pred)

        bias = np.mean(y_pred - y_true)

        # === Store and Print Results ===
        metrics['MSE'][ticker] = mse
        metrics['MAE'][ticker] = mae
        metrics['RMSE'][ticker] = rmse
        metrics['Error Rate (%)'][ticker] = error_rate
        metrics['MAPE (%)'][ticker] = mape
        metrics['WQL'][ticker] = wql
        metrics['Theil\'s U'][ticker] = theils_u
        metrics['Pearson'][ticker] = pearson_corr
        metrics['Bias'][ticker] = bias

        processed += 1
        print(f"{ticker}:"
              f" MSE = {safe_float_format(mse)},"
              f" MAE = {safe_float_format(mae)},"
              f" RMSE = {safe_float_format(rmse)},"
              f" Error Rate (%) = {safe_float_format(error_rate, '.2f')},"
              f" MAPE (%) = {safe_float_format(mape, '.2f')},"
              f" WQL = {safe_float_format(wql)},"
              f" Theil's U = {safe_float_format(theils_u)},"
              f" Pearson = {safe_float_format(pearson_corr)},"
              f" Bias = {safe_float_format(bias)}")

    except Exception as e:
        print(f"Error processing {ticker}: {str(e)}")
        skipped += 1
        for metric in metrics:
            metrics[metric][ticker] = np.nan
        continue

print(f"\nCompanies processed: {processed}")
print(f"Companies skipped: {skipped}")

# === Print Average Metrics ===
if processed > 0:
    print("\n📊 Average Metrics Across Companies:")
    for metric in metrics:
        valid = [v for v in metrics[metric].values() if ~np.isnan(v)]
        avg = np.mean(valid) if valid else np.nan
        print(f"{metric}: {safe_float_format(avg)}")

# === Save Results ===
results_df = pd.DataFrame(metrics)
results_df.to_excel("chronos_metrics_results_window8.xlsx", index=True)
print("\n📁 Results saved to Excel")


Total companies loaded: 495
A: MSE = 15.0410, MAE = 2.9377, RMSE = 3.8783, Error Rate (%) = 24.47, MAPE (%) = 24.47, WQL = 2.3414, Theil's U = 0.1201, Pearson = NaN, Bias = 2.9355
AAPL: MSE = 185.5022, MAE = 8.0085, RMSE = 13.6199, Error Rate (%) = 47.32, MAPE (%) = 47.32, WQL = 24.2347, Theil's U = 0.2911, Pearson = -0.1062, Bias = 8.0085
ABBV: MSE = 109.7821, MAE = 5.8029, RMSE = 10.4777, Error Rate (%) = 20.64, MAPE (%) = 20.64, WQL = 23.8339, Theil's U = 0.1733, Pearson = 0.0830, Bias = 2.4862
ABT: MSE = 235.2716, MAE = 9.8419, RMSE = 15.3386, Error Rate (%) = 39.31, MAPE (%) = 39.31, WQL = 17.7254, Theil's U = 0.2440, Pearson = 0.0713, Bias = 7.1812
ACGL: MSE = 94.0106, MAE = 6.6421, RMSE = 9.6959, Error Rate (%) = 30.61, MAPE (%) = 30.61, WQL = 16.3813, Theil's U = 0.1855, Pearson = 0.0892, Bias = 6.4724
ACN: MSE = 543.0257, MAE = 12.0376, RMSE = 23.3029, Error Rate (%) = 111.31, MAPE (%) = 111.31, WQL = 237.3057, Theil's U = 0.5536, Pearson = -0.1472, Bias = 11.9825
ADBE: MSE = 

withour pearson

In [ ]:
import ast
import numpy as np
from sklearn.metrics import mean_squared_error, mean_absolute_error
import pandas as pd
from chronos import ChronosPipeline
import torch


# === Load and Parse the ESG Data ===
with open('/content/PaperReady_numeric_esg.txt', 'r') as f:
    content = f.read()
    start = content.find('{')
    end = content.rfind('}') + 1
    dict_str = content[start:end]
    numeric_esg = ast.literal_eval(dict_str)


# === Load Chronos Model ===
pretrained_model = ChronosPipeline.from_pretrained("amazon/chronos-t5-small", device_map="auto")


# === Define Metrics Storage ===
metrics = {
    'MSE': {}, 'MAE': {}, 'RMSE': {},
    'Error Rate (%)': {}, 'MAPE (%)': {}, 'WQL': {},
    "Theil's U": {}, "Bias": {}
}
wql_var_threshold = 1e-6
window_size = 8


# === Loop Through Each Company ===
for ticker, series in numeric_esg.items():
    if len(series) < window_size * 2:
        for metric in metrics:
            metrics[metric][ticker] = np.nan
        continue

    series = np.array(series, dtype=float)
    n = len(series)
    split_idx = int(n * 0.75)
    train, test = series[:split_idx], series[split_idx:]

    # Use sliding window on training data
    input_windows = []
    for i in range(len(train) - window_size + 1):
        input_windows.append(train[i:i + window_size])
    if not input_windows:
        for metric in metrics:
            metrics[metric][ticker] = np.nan
        continue

    # Use the last window from training data for prediction
    input_tensor = torch.tensor(input_windows[-1], dtype=torch.float32).unsqueeze(0)
    prediction_length = len(test)

    # === Predict ===
    forecast = pretrained_model.predict(input_tensor, prediction_length=prediction_length)
    y_pred = forecast[0, :, 0].detach().cpu().numpy()
    y_true = test

    min_len = min(len(y_pred), len(y_true))
    y_pred = y_pred[:min_len]
    y_true = y_true[:min_len]

    # === Metrics Calculation ===
    mse = mean_squared_error(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    abs_true = np.abs(y_true)
    error_rate = 100 * np.mean(np.abs(y_true - y_pred) / np.where(abs_true == 0, np.nan, abs_true))
    mape = 100 * np.mean(np.abs((y_true - y_pred) / np.where(y_true == 0, np.nan, y_true)))
    variance = np.var(y_true)
    wql = np.nan if variance < wql_var_threshold else mse / variance

    # --- Theil's U calculation ---
    denom = np.sqrt(np.sum(y_pred ** 2)) + np.sqrt(np.sum(y_true ** 2))
    theils_u = np.nan if denom == 0 else np.sqrt(np.sum((y_pred - y_true) ** 2)) / denom

    # --- Bias calculation ---
    bias = np.mean(y_pred - y_true)

    # === Store Metrics ===
    metrics['MSE'][ticker] = mse
    metrics['MAE'][ticker] = mae
    metrics['RMSE'][ticker] = rmse
    metrics['Error Rate (%)'][ticker] = error_rate
    metrics['MAPE (%)'][ticker] = mape
    metrics['WQL'][ticker] = wql
    metrics["Theil's U"][ticker] = theils_u
    metrics["Bias"][ticker] = bias

    print(f"{ticker}: MSE = {mse:.4f}, MAE = {mae:.4f}, RMSE = {rmse:.4f}, "
          f"Error Rate (%) = {error_rate:.2f}, MAPE (%) = {mape:.2f}, WQL = {wql if not np.isnan(wql) else 'NaN'}, "
          f"Theil's U = {theils_u if not np.isnan(theils_u) else 'NaN'}, Bias = {bias:.4f}")


# === Print Average Metrics Across All Companies ===
print("\n📊 Average Metrics Across Companies:")
avg_metrics = {}
for metric, values in metrics.items():
    valid = [v for v in values.values() if not np.isnan(v)]
    avg = np.mean(valid) if valid else np.nan
    avg_metrics[metric] = avg
    if metric != "RMSE":
        print(f"{metric}: {avg:.4f}")

# Calculate overall RMSE as sqrt of avg MSE
if not np.isnan(avg_metrics["MSE"]):
    avg_rmse = np.sqrt(avg_metrics["MSE"])
else:
    avg_rmse = np.nan
print(f"RMSE: {avg_rmse:.4f}")


# === Save Results to Excel ===
results_df = pd.DataFrame(metrics)
results_df.to_excel("chronos_metrics_results_window8.xlsx", index=True)
print("\n📁 Results saved to 'chronos_metrics_results_window8.xlsx'")


A: MSE = 19.5506, MAE = 3.2932, RMSE = 4.4216, Error Rate (%) = 27.28, MAPE (%) = 27.28, WQL = 3.043426901710684, Theil's U = 0.138318716624035, Bias = 2.1998
AAPL: MSE = 182.8862, MAE = 7.9545, RMSE = 13.5235, Error Rate (%) = 47.32, MAPE (%) = 47.32, WQL = 23.892966934734925, Theil's U = 0.289907430358942, Bias = 7.9299
ABBV: MSE = 321.6843, MAE = 10.7947, RMSE = 17.9356, Error Rate (%) = 38.51, MAPE (%) = 38.51, WQL = 69.83826781261945, Theil's U = 0.25857734838017915, Bias = 10.2522
ABT: MSE = 32272.7672, MAE = 46.1743, RMSE = 179.6462, Error Rate (%) = 205.33, MAPE (%) = 205.33, WQL = 2431.431315451218, Theil's U = 0.8406408869861816, Bias = 44.5716
ACGL: MSE = 47.0398, MAE = 4.7244, RMSE = 6.8586, Error Rate (%) = 22.43, MAPE (%) = 22.43, WQL = 8.196657177908941, Theil's U = 0.13773278185458257, Bias = 4.5050
ACN: MSE = 143.2162, MAE = 6.1629, RMSE = 11.9673, Error Rate (%) = 56.50, MAPE (%) = 56.50, WQL = 62.58638222252533, Theil's U = 0.37715976293132353, Bias = 6.1078
ADBE: MS

In [ ]:
import ast
import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error, mean_absolute_error
from chronos import ChronosPipeline
import torch

# === Set Random Seed for Reproducibility ===
seed = 42
np.random.seed(seed)
torch.manual_seed(seed)

# === Load and Parse the ESG Data ===
with open('/content/PaperReady_numeric_esg.txt', 'r') as f:
    content = f.read()
    start = content.find('{')
    end = content.rfind('}') + 1
    dict_str = content[start:end]
    numeric_esg = ast.literal_eval(dict_str)

# === Load Chronos Model ===
pretrained_model = ChronosPipeline.from_pretrained("amazon/chronos-t5-small", device_map="auto")

# === Define Hyperparameters ===
window_size = 8  # Fixed window size
# You can define other hyperparameters here if needed

# === Define Metrics Storage ===
metrics = {
    'MSE': {}, 'MAE': {}, 'MAPE (%)': {},
    "Theil's U": {}, "Bias": {}
}

# === Loop Through Each Company ===
for ticker, series in numeric_esg.items():
    if len(series) < window_size * 2:
        for metric in metrics:
            metrics[metric][ticker] = np.nan
        continue

    series = np.array(series, dtype=float)
    n = len(series)
    split_idx = int(n * 0.75)
    train, test = series[:split_idx], series[split_idx:]

    # Sliding window on training data
    input_windows = []
    for i in range(len(train) - window_size + 1):
        input_windows.append(train[i:i + window_size])
    if not input_windows:
        for metric in metrics:
            metrics[metric][ticker] = np.nan
        continue

    # Last window for prediction
    input_tensor = torch.tensor(input_windows[-1], dtype=torch.float32).unsqueeze(0)
    prediction_length = len(test)

    # Predict
    forecast = pretrained_model.predict(input_tensor, prediction_length=prediction_length)
    y_pred = forecast[0, :, 0].detach().cpu().numpy()
    y_true = test

    # Align prediction length
    min_len = min(len(y_pred), len(y_true))
    y_pred = y_pred[:min_len]
    y_true = y_true[:min_len]

    # === Metrics Calculation ===
    mse = mean_squared_error(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    mape = 100 * np.mean(np.abs((y_true - y_pred) / np.where(y_true == 0, np.nan, y_true)))

    denom = np.sqrt(np.sum(y_pred ** 2)) + np.sqrt(np.sum(y_true ** 2))
    theils_u = np.nan if denom == 0 else np.sqrt(np.sum((y_pred - y_true) ** 2)) / denom

    bias = np.mean(y_pred - y_true)

    # === Store Metrics ===
    metrics['MSE'][ticker] = mse
    metrics['MAE'][ticker] = mae
    metrics['MAPE (%)'][ticker] = mape
    metrics["Theil's U"][ticker] = theils_u
    metrics["Bias"][ticker] = bias

    print(f"{ticker}: MSE = {mse:.4f}, MAE = {mae:.4f}, MAPE = {mape:.2f}%, "
          f"Theil's U = {theils_u:.4f}, Bias = {bias:.4f}")

# === Print Average Metrics Across All Companies ===
print("\n📊 Average Metrics Across Companies:")
avg_metrics = {}
for metric, values in metrics.items():
    valid = [v for v in values.values() if not np.isnan(v)]
    avg = np.mean(valid) if valid else np.nan
    avg_metrics[metric] = avg
    print(f"{metric}: {avg:.4f}")

# === Save Results to Excel ===
results_df = pd.DataFrame(metrics)
results_df.to_excel("chronos_selected_metrics.xlsx", index=True)
print("\n📁 Results saved to 'chronos_selected_metrics.xlsx'")


A: MSE = 403.2719, MAE = 10.2550, MAPE = 67.71%, Theil's U = 0.4419, Bias = 10.2528
AAPL: MSE = 343.2775, MAE = 10.8813, MAPE = 63.56%, Theil's U = 0.3629, Bias = 10.8767
ABBV: MSE = 81.2253, MAE = 3.9068, MAPE = 14.37%, Theil's U = 0.1480, Bias = 3.5602
ABT: MSE = 151.8143, MAE = 7.2070, MAPE = 29.97%, Theil's U = 0.2099, Bias = 4.0385
ACGL: MSE = 18.1389, MAE = 3.7407, MAPE = 17.77%, Theil's U = 0.0879, Bias = 3.5214
ACN: MSE = 229.6316, MAE = 7.5286, MAPE = 68.84%, Theil's U = 0.4401, Bias = 7.4735
ADBE: MSE = 156.2944, MAE = 3.9079, MAPE = 29.50%, Theil's U = 0.3764, Bias = 3.4124
ADI: MSE = 72.1519, MAE = 4.9784, MAPE = 25.56%, Theil's U = 0.1690, Bias = 4.1250
ADM: MSE = 152.7049, MAE = 7.2521, MAPE = 20.95%, Theil's U = 0.1881, Bias = -4.6130
ADP: MSE = 13.4170, MAE = 1.8580, MAPE = 13.45%, Theil's U = 0.1285, Bias = 0.2637
ADSK: MSE = 142.9198, MAE = 6.2440, MAPE = 39.40%, Theil's U = 0.3027, Bias = 4.2855
AEE: MSE = 140.8210, MAE = 8.0587, MAPE = 27.35%, Theil's U = 0.1697, Bi